In [1]:
"""
Model Quantization


 Dynamic Quantization — one line of code, no model changes
 Static Quantization  — needs a quantizable model wrapper, but better results


Paper of the research  "Quantization and Training of Neural Networks for Efficient
Integer-Arithmetic-Only Inference" , Benoit Jacob et al. (2018)
  https://arxiv.org/abs/1712.05877
"""

import torch
import torch.nn as nn
import torch.quantization as quant
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import os




In [2]:
# @title
class VGGStyleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # ── Block 1: 3 → 64 → 64, pool ──
        self.conv1a = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        # ── Block 2: 64 → 128 → 128, pool ──
        self.conv2a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        # ── Block 3: 128 → 256 → 256, pool ──
        self.conv3a = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        # ── Block 4: 256 → 512 → 512, pool ──
        self.conv4a = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4a = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4b = nn.BatchNorm2d(512)

        # ── Classifier ──
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, num_classes)

        # Initialize weights (He initialization, same as your original)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)   # gamma = 1
                nn.init.zeros_(m.bias)    # beta = 0
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Block 1: (N,3,32,32) → (N,64,16,16)
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.1, training=self.training)

        # Block 2: (N,64,16,16) → (N,128,8,8)
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.2, training=self.training)

        # Block 3: (N,128,8,8) → (N,256,4,4)
        x = F.relu(self.bn3a(self.conv3a(x)))
        x = F.relu(self.bn3b(self.conv3b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.3, training=self.training)

        # Block 4: (N,256,4,4) → (N,512,2,2)
        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.4, training=self.training)

        # GAP: (N,512,2,2) → (N,512)
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)

        # Classifier
        x = F.relu(self.fc1(x))
        x = F.dropout(x, 0.5, training=self.training)
        x = self.fc2(x)          # raw logits — use nn.CrossEntropyLoss

        return x



In [3]:
BATCH_SIZE = 128




transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)


100%|██████████| 170M/170M [00:03<00:00, 51.1MB/s]


In [4]:


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to('cpu'), labels.to('cpu')
            logits = model(inputs)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def get_model_size_mb(model):
    tmp_path = "/tmp/_temp_model.pth"
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    os.remove(tmp_path)
    return size_mb


def measure_latency(model, num_runs=200):
    model.eval()
    dummy = torch.randn(1, 3, 32, 32)
    # Warm-up
    for _ in range(20):
        with torch.no_grad():
            model(dummy)
    # Timed
    start = time.time()
    for _ in range(num_runs):
        with torch.no_grad():
            model(dummy)
    avg_ms = (time.time() - start) / num_runs * 1000
    return avg_ms



In [5]:


# Base  FP32
# ───────────────────────────────────────────
def run_baseline(model_path):
    print("\n" + "=" * 60)
    print("  Baseline FP32")
    print("=" * 60)

    model = VGGStyleCNN()
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.to('cpu').eval()

    test_acc = evaluate(model, test_loader)
    size_mb = get_model_size_mb(model)
    latency = measure_latency(model)

    print(f"  Test Accuracy: {test_acc:.2f}%")
    print(f"  Model Size:    {size_mb:.2f} MB")
    print(f"  Latency:       {latency:.2f} ms")

    return {'method': 'FP32 (32-bit)', 'bits': 32,
            'test_acc': test_acc, 'size_mb': size_mb, 'latency_ms': latency}



In [6]:

#Dynamic Quantization

def run_dynamic_quantization(model_path):
    """
    Quantizes Linear layer weights to INT8 at save time.
    Activations are quantized during inference.
 torch.quantization.quantize_dynamic()


    only quantizes Linear (FC) layers, not Conv2d.


    """
    print("\n" + "=" * 60)
    print("  Dynamic Quantization (INT8)")
    print("=" * 60)

    model = VGGStyleCNN()
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.to('cpu').eval()

    #  quantizes  Linear layers to INT8
    model_quantized = torch.quantization.quantize_dynamic(
        model,
        {nn.Linear},       # the layers to quantize
        dtype=torch.qint8  #
    )

    test_acc = evaluate(model_quantized, test_loader)
    size_mb = get_model_size_mb(model_quantized)
    latency = measure_latency(model_quantized)

    print(f"  Test Accuracy: {test_acc:.2f}%")
    print(f"  Model Size:    {size_mb:.2f} MB")
    print(f"  Latency:       {latency:.2f} ms")

    return {'method': 'Dynamic (INT8)', 'bits': 8,
            'test_acc': test_acc, 'size_mb': size_mb, 'latency_ms': latency}


In [7]:

#Static Quantization

class VGGQuantizable(nn.Module):
    """
    Thin wrapper that adds QuantStub/DeQuantStub and
    uses nn.ReLU modules (needed for Conv+BN+ReLU fusion).
    Same layers and sizes as VGGStyleCNN.
    """
    def __init__(self):
        super().__init__()
        self.quant = quant.QuantStub()
        self.dequant = quant.DeQuantStub()



        self.conv1a = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.relu1a = nn.ReLU()
        self.conv1b = nn.Conv2d(64, 64, 3, padding=1)
        self.bn1b = nn.BatchNorm2d(64)
        self.relu1b = nn.ReLU()

        self.conv2a = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.relu2a = nn.ReLU()
        self.conv2b = nn.Conv2d(128, 128, 3, padding=1)
        self.bn2b = nn.BatchNorm2d(128)
        self.relu2b = nn.ReLU()



        self.conv3a = nn.Conv2d(128, 256, 3, padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.relu3a = nn.ReLU()

        self.conv3b = nn.Conv2d(256, 256, 3, padding=1)
        self.bn3b = nn.BatchNorm2d(256)
        self.relu3b = nn.ReLU()



        self.conv4a = nn.Conv2d(256, 512, 3, padding=1)
        self.bn4a = nn.BatchNorm2d(512)
        self.relu4a = nn.ReLU()
        self.conv4b = nn.Conv2d(512, 512, 3, padding=1)
        self.bn4b = nn.BatchNorm2d(512)
        self.relu4b = nn.ReLU()




        self.fc1 = nn.Linear(512, 256)
        self.relu_fc = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)








    def forward(self, x):
        x = self.quant(x)




        x = self.relu1a(self.bn1a(self.conv1a(x)))
        x = self.relu1b(self.bn1b(self.conv1b(x)))
        x = nn.functional.max_pool2d(x, 2)

        x = self.relu2a(self.bn2a(self.conv2a(x)))
        x = self.relu2b(self.bn2b(self.conv2b(x)))
        x = nn.functional.max_pool2d(x, 2)

        x = self.relu3a(self.bn3a(self.conv3a(x)))
        x = self.relu3b(self.bn3b(self.conv3b(x)))
        x = nn.functional.max_pool2d(x, 2)

        x = self.relu4a(self.bn4a(self.conv4a(x)))
        x = self.relu4b(self.bn4b(self.conv4b(x)))
        x = nn.functional.max_pool2d(x, 2)




        x = nn.functional.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)

        x = self.dequant(x)
        return x

    def fuse_model(self):


        torch.quantization.fuse_modules(self, [
            ['conv1a', 'bn1a', 'relu1a'], ['conv1b', 'bn1b', 'relu1b'],
            ['conv2a', 'bn2a', 'relu2a'], ['conv2b', 'bn2b', 'relu2b'],
            ['conv3a', 'bn3a', 'relu3a'], ['conv3b', 'bn3b', 'relu3b'],
            ['conv4a', 'bn4a', 'relu4a'], ['conv4b', 'bn4b', 'relu4b'],
            ['fc1', 'relu_fc'],
        ], inplace=True)




















def run_static_quantization(model_path):
    """
    Static quantization — quantizes  weights & activations.
    load weights then fuse layers then run on real data then convert to INT8.
    """
    print("\n" + "=" * 60)

    print("  Static Quantization (INT8)")

    print("=" * 60)

    # Load weights into quantizable wrapper
    model = VGGQuantizable()

    original_weights = torch.load(model_path, map_location='cpu')
    model_dict = model.state_dict()

    matched = {k: v for k, v in original_weights.items() if k in model_dict}
    model_dict.update(matched)
    model.load_state_dict(model_dict)
    print(f"  Loaded {len(matched)} tensors from trained model")

    model.eval()
    model.fuse_model()
    print("  Fused Conv+BN+ReLU")

    # observers and calibrate
    model.qconfig = quant.get_default_qconfig('fbgemm')
    quant.prepare(model, inplace=True)


    with torch.no_grad():
        for i, (inputs, _) in enumerate(train_loader):
            if i >= 50:
                break
            model(inputs)

    # Convert to INT8
    quant.convert(model, inplace=True)
    print("  Converted to INT8")

    test_acc = evaluate(model, test_loader)
    size_mb = get_model_size_mb(model)
    latency = measure_latency(model)

    print(f"  Test Accuracy: {test_acc:.2f}%")
    print(f"  Model Size:    {size_mb:.2f} MB")
    print(f"  Latency:       {latency:.2f} ms")

    return {'method': 'Static (INT8)', 'bits': 8,
            'test_acc': test_acc, 'size_mb': size_mb, 'latency_ms': latency}



In [8]:

# Visualization + Summary

def plot_results(results):
    os.makedirs("plots", exist_ok=True)

    methods = [r['method'] for r in results]
    accs = [r['test_acc'] for r in results]
    sizes = [r['size_mb'] for r in results]
    latencies = [r['latency_ms'] for r in results]

    colors = ['#2196F3', '#FF9800', '#4CAF50']

    # ── Accuracy vs                       bit-width ──
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, r in enumerate(results):
        ax.scatter(r['bits'], r['test_acc'], s=150, c=colors[i],
                   zorder=5, label=r['method'])
        ax.annotate(f"{r['test_acc']:.1f}%", (r['bits'], r['test_acc']),
                    textcoords="offset points", xytext=(0, 15),
                    ha='center', fontsize=11)
    ax.set_xlabel('Bit-width', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Quantization: Accuracy vs Bit-width', fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/quant_accuracy_vs_bits.png", dpi=150)
    plt.close()

    # ── Size comparison ──
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(methods, sizes, color=colors, width=0.5)
    for bar, s in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{s:.1f} MB', ha='center', fontsize=11)
    ax.set_ylabel('Model Size (MB)', fontsize=12)
    ax.set_title('Model Size Comparison', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig("plots/quant_model_size.png", dpi=150)
    plt.close()










    #Latency comparison
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(methods, latencies, color=colors, width=0.5)
    for bar, l in zip(bars, latencies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{l:.1f} ms', ha='center', fontsize=11)
    ax.set_ylabel('Latency (ms)', fontsize=12)
    ax.set_title('Inference Latency (CPU, batch=1)', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig("plots/quant_latency.png", dpi=150)
    plt.close()

    print("Saved plots to plots/ directory")













def print_summary(results):
    baseline = results[0]
    print(f"\n{'='*72}")
    print(f"{'Method':<20} | {'Bits':>5} | {'Acc':>7} | {'Size':>8} | {'Latency':>10}")
    print(f"{'-'*72}")
    for r in results:
        print(f"{r['method']:<20} | {r['bits']:>5} | {r['test_acc']:>6.2f}% | "
              f"{r['size_mb']:>6.2f} MB | {r['latency_ms']:>8.2f} ms")
    print(f"{'='*72}")

    print(f"\nCompression vs baseline:")
    for r in results[1:]:
        ratio = baseline['size_mb'] / r['size_mb']
        speedup = baseline['latency_ms'] / r['latency_ms']
        acc_diff = r['test_acc'] - baseline['test_acc']
        print(f"  {r['method']}: {ratio:.1f}x smaller, "
              f"{speedup:.2f}x faster, {acc_diff:+.2f}% accuracy")



In [9]:



# Run

if __name__ == "__main__":
    MODEL_PATH = "vgg_cifar10_trained.pth"

    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: {MODEL_PATH} not found! Run train.py first.")
        exit(1)

    results = []
    results.append(run_baseline(MODEL_PATH))
    results.append(run_dynamic_quantization(MODEL_PATH))
    results.append(run_static_quantization(MODEL_PATH))

    print_summary(results)
    plot_results(results)

ERROR: vgg_cifar10_trained.pth not found! Run train.py first.

  Baseline FP32


FileNotFoundError: [Errno 2] No such file or directory: 'vgg_cifar10_trained.pth'

In [ ]:
    plot_results(results)
